# GRUPA 3 - Vision Transformer vs CNN

## Analiza porównawcza architektur na CIFAR-10 i ImageNet

Projekt zawiera badanie różnic między klasyczną architekturą CNN a nowoczesnym Vision Transformerem (ViT) na dwóch zbiorach danych: CIFAR-10 i ImageNet.

**Główne pytania badawcze:**
1. Jak różnią się dokładności klasyfikacji CNN i ViT?
2. Jak różnią się czasy treningu i liczba parametrów?
3. Które podejście jest bardziej efektywne na każdym zbiorze danych?
4. Jak skaluje się wydajność wraz ze wzrostem złożoności modelu?

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Konfiguracja stylu wykreśów
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Wczytanie danych
with open('porownanie_wynikow.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print('Dane załadowane pomyślnie!')
print(f'Dostępne datasety: {list(data.keys())}')

## 1. Przygotowanie danych

In [ ]:
# Funkcja do ekstraktowania danych z JSON-a (NAPRAWIONA)
def extract_results(data_dict):
    results = []
    for item in data_dict:
        # Oblicz time_per_epoch jeśli brakuje
        num_epochs = len(item.get('history', []))
        if num_epochs == 0:
            num_epochs = item.get('param_value', 1)
        
        time_per_epoch = item.get('time_per_epoch')
        if time_per_epoch is None:
            time_per_epoch = item['train_time_s'] / num_epochs
        
        results.append({
            'Model': item['model'],
            'Experiment': item['experiment'],
            'Test Accuracy (%)': item['test_acc'],
            'Train Accuracy (%)': item['train_acc'],
            'Training Time (s)': item['train_time_s'],
            'Liczba Parametrów': item['num_params'],
            'Czas/Epoch (s)': time_per_epoch
        })
    return pd.DataFrame(results)

# Wczytaj dane dla obu datasetów
cifar10_df = extract_results(data['cifar10'])
imagenet_df = extract_results(data['imagenet'])

print('CIFAR-10 Wyniki:')
print(cifar10_df)
print('\n' + '='*80 + '\n')
print('ImageNet Wyniki:')
print(imagenet_df)

## 2. Porównanie Dokładności - CIFAR-10

In [ ]:
# Wykres dokładności na CIFAR-10
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test Accuracy
colors = ['#FF6B6B' if model == 'CNN' else '#4ECDC4' for model in cifar10_df['Model']]
axes[0].bar(range(len(cifar10_df)), cifar10_df['Test Accuracy (%)'], color=colors, alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Eksperyment', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Test Accuracy (%)', fontsize=11, fontweight='bold')
axes[0].set_title('CIFAR-10: Test Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(len(cifar10_df)))
axes[0].set_xticklabels(cifar10_df['Experiment'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Dodaj wartości na słupkach
for i, v in enumerate(cifar10_df['Test Accuracy (%)']):
    axes[0].text(i, v + 1, f'{v:.2f}%', ha='center', va='bottom', fontweight='bold')

# Train Accuracy
axes[1].bar(range(len(cifar10_df)), cifar10_df['Train Accuracy (%)'], color=colors, alpha=0.8, edgecolor='black')
axes[1].set_xlabel('Eksperyment', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Train Accuracy (%)', fontsize=11, fontweight='bold')
axes[1].set_title('CIFAR-10: Train Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(len(cifar10_df)))
axes[1].set_xticklabels(cifar10_df['Experiment'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

# Dodaj wartości na słupkach
for i, v in enumerate(cifar10_df['Train Accuracy (%)']):
    axes[1].text(i, v + 1, f'{v:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('01_cifar10_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n📊 Statystyka CIFAR-10:')
print(f'Best CNN Test Accuracy: {cifar10_df[cifar10_df["Model"] == "CNN"]["Test Accuracy (%)"].max():.2f}%')
print(f'Best ViT Test Accuracy: {cifar10_df[cifar10_df["Model"] == "ViT"]["Test Accuracy (%)"].max():.2f}%')

## 3. Porównanie Dokładności - ImageNet

In [ ]:
# Wykres dokładności na ImageNet
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test Accuracy
colors = ['#FF6B6B' if model == 'CNN' else '#4ECDC4' for model in imagenet_df['Model']]
axes[0].bar(range(len(imagenet_df)), imagenet_df['Test Accuracy (%)'], color=colors, alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Eksperyment', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Test Accuracy (%)', fontsize=11, fontweight='bold')
axes[0].set_title('ImageNet: Test Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(len(imagenet_df)))
axes[0].set_xticklabels(imagenet_df['Experiment'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Dodaj wartości na słupkach
for i, v in enumerate(imagenet_df['Test Accuracy (%)']):
    axes[0].text(i, v + 0.5, f'{v:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)

# Train Accuracy
axes[1].bar(range(len(imagenet_df)), imagenet_df['Train Accuracy (%)'], color=colors, alpha=0.8, edgecolor='black')
axes[1].set_xlabel('Eksperyment', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Train Accuracy (%)', fontsize=11, fontweight='bold')
axes[1].set_title('ImageNet: Train Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(len(imagenet_df)))
axes[1].set_xticklabels(imagenet_df['Experiment'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

# Dodaj wartości na słupkach
for i, v in enumerate(imagenet_df['Train Accuracy (%)']):
    axes[1].text(i, v + 0.5, f'{v:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('02_imagenet_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n📊 Statystyka ImageNet:')
print(f'Best CNN Test Accuracy: {imagenet_df[imagenet_df["Model"] == "CNN"]["Test Accuracy (%)"].max():.2f}%')
print(f'Best ViT Test Accuracy: {imagenet_df[imagenet_df["Model"] == "ViT"]["Test Accuracy (%)"].max():.2f}%')

## 4. Porównanie Czasu Treningu

In [ ]:
# Porównanie czasu treningu
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CIFAR-10
colors_cifar = ['#FF6B6B' if model == 'CNN' else '#4ECDC4' for model in cifar10_df['Model']]
axes[0].bar(range(len(cifar10_df)), cifar10_df['Training Time (s)'], color=colors_cifar, alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Eksperyment', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Czas Treningu (s)', fontsize=11, fontweight='bold')
axes[0].set_title('CIFAR-10: Czas Treningu', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(len(cifar10_df)))
axes[0].set_xticklabels(cifar10_df['Experiment'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Dodaj wartości
for i, v in enumerate(cifar10_df['Training Time (s)']):
    axes[0].text(i, v + 30, f'{v:.0f}s', ha='center', va='bottom', fontweight='bold', fontsize=9)

# ImageNet
colors_imgnet = ['#FF6B6B' if model == 'CNN' else '#4ECDC4' for model in imagenet_df['Model']]
axes[1].bar(range(len(imagenet_df)), imagenet_df['Training Time (s)'], color=colors_imgnet, alpha=0.8, edgecolor='black')
axes[1].set_xlabel('Eksperyment', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Czas Treningu (s)', fontsize=11, fontweight='bold')
axes[1].set_title('ImageNet: Czas Treningu', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(len(imagenet_df)))
axes[1].set_xticklabels(imagenet_df['Experiment'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

# Dodaj wartości
for i, v in enumerate(imagenet_df['Training Time (s)']):
    axes[1].text(i, v + 10, f'{v:.0f}s', ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('03_training_time.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n⏱️ Statystyka Czasu Treningu:')
print(f'\nCIFAR-10:')
print(f'  CNN średni czas: {cifar10_df[cifar10_df["Model"] == "CNN"]["Training Time (s)"].mean():.2f}s')
print(f'  ViT średni czas: {cifar10_df[cifar10_df["Model"] == "ViT"]["Training Time (s)"].mean():.2f}s')
print(f'\nImageNet:')
print(f'  CNN średni czas: {imagenet_df[imagenet_df["Model"] == "CNN"]["Training Time (s)"].mean():.2f}s')
print(f'  ViT średni czas: {imagenet_df[imagenet_df["Model"] == "ViT"]["Training Time (s)"].mean():.2f}s')

## 5. Podsumowanie Wyników

In [ ]:
# Oblicze statystyki podsumowujące
print('\n' + '='*80)
print('PODSUMOWANIE WYNIKÓW')
print('='*80)

print('\n📊 BEST MODELS - CIFAR-10')
print('-' * 80)
best_cifar10_cnn = cifar10_df[cifar10_df['Model'] == 'CNN'].loc[cifar10_df[cifar10_df['Model'] == 'CNN']['Test Accuracy (%)'].idxmax()]
best_cifar10_vit = cifar10_df[cifar10_df['Model'] == 'ViT'].loc[cifar10_df[cifar10_df['Model'] == 'ViT']['Test Accuracy (%)'].idxmax()]

print(f'\nBest CNN:')
print(f'  Model: {best_cifar10_cnn["Experiment"]}')
print(f'  Test Accuracy: {best_cifar10_cnn["Test Accuracy (%)"]:.2f}%')
print(f'  Parameters: {best_cifar10_cnn["Liczba Parametrów"]/1e6:.2f}M')
print(f'  Training Time: {best_cifar10_cnn["Training Time (s)"]:.2f}s')

print(f'\nBest ViT:')
print(f'  Model: {best_cifar10_vit["Experiment"]}')
print(f'  Test Accuracy: {best_cifar10_vit["Test Accuracy (%)"]:.2f}%')
print(f'  Parameters: {best_cifar10_vit["Liczba Parametrów"]/1e6:.2f}M')
print(f'  Training Time: {best_cifar10_vit["Training Time (s)"]:.2f}s')

print(f'\n✨ Zwycięzca CIFAR-10: {"CNN" if best_cifar10_cnn["Test Accuracy (%)"] > best_cifar10_vit["Test Accuracy (%)"] else "ViT"}')
print(f'  Różnica: {abs(best_cifar10_cnn["Test Accuracy (%)"] - best_cifar10_vit["Test Accuracy (%)"]):.2f}%')

print('\n\n📊 BEST MODELS - ImageNet')
print('-' * 80)
best_imagenet_cnn = imagenet_df[imagenet_df['Model'] == 'CNN'].loc[imagenet_df[imagenet_df['Model'] == 'CNN']['Test Accuracy (%)'].idxmax()]
best_imagenet_vit = imagenet_df[imagenet_df['Model'] == 'ViT'].loc[imagenet_df[imagenet_df['Model'] == 'ViT']['Test Accuracy (%)'].idxmax()]

print(f'\nBest CNN:')
print(f'  Model: {best_imagenet_cnn["Experiment"]}')
print(f'  Test Accuracy: {best_imagenet_cnn["Test Accuracy (%)"]:.2f}%')
print(f'  Parameters: {best_imagenet_cnn["Liczba Parametrów"]/1e6:.2f}M')
print(f'  Training Time: {best_imagenet_cnn["Training Time (s)"]:.2f}s')

print(f'\nBest ViT:')
print(f'  Model: {best_imagenet_vit["Experiment"]}')
print(f'  Test Accuracy: {best_imagenet_vit["Test Accuracy (%)"]:.2f}%')
print(f'  Parameters: {best_imagenet_vit["Liczba Parametrów"]/1e6:.2f}M')
print(f'  Training Time: {best_imagenet_vit["Training Time (s)"]:.2f}s')

print(f'\n✨ Zwycięzca ImageNet: {"CNN" if best_imagenet_cnn["Test Accuracy (%)"] > best_imagenet_vit["Test Accuracy (%)"] else "ViT"}')
print(f'  Różnica: {abs(best_imagenet_cnn["Test Accuracy (%)"] - best_imagenet_vit["Test Accuracy (%)"]):.2f}%')

print('\n' + '='*80)

## 6. Krzywe Treningu - CIFAR-10

In [ ]:
# Wczytaj dane historii dla CIFAR-10
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('CIFAR-10: Krzywe Treningu', fontsize=14, fontweight='bold', y=1.00)

for idx, (ax, item) in enumerate(zip(axes.flat, data['cifar10'])):
    history = item['history']
    epochs = [h['epoch'] for h in history]
    train_acc = [h['train_acc'] for h in history]
    test_acc = [h['test_acc'] for h in history]
    loss = [h['loss'] for h in history]
    
    ax2 = ax.twinx()
    
    # Dokładność
    ax.plot(epochs, train_acc, 'o-', label='Train Acc', color='#2E86AB', linewidth=2, markersize=6)
    ax.plot(epochs, test_acc, 's-', label='Test Acc', color='#A23B72', linewidth=2, markersize=6)
    ax.set_ylabel('Accuracy (%)', fontsize=10, fontweight='bold', color='#2E86AB')
    ax.tick_params(axis='y', labelcolor='#2E86AB')
    
    # Loss
    ax2.plot(epochs, loss, '^-', label='Loss', color='#F18F01', linewidth=2, markersize=6)
    ax2.set_ylabel('Loss', fontsize=10, fontweight='bold', color='#F18F01')
    ax2.tick_params(axis='y', labelcolor='#F18F01')
    
    ax.set_xlabel('Epoch', fontsize=10, fontweight='bold')
    ax.set_title(f'{item["model"]} - {item["experiment"]}', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)
    
    # Legenda
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='right', fontsize=8)

plt.tight_layout()
plt.savefig('04_cifar10_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Krzywe Treningu - ImageNet

In [ ]:
# Wczytaj dane historii dla ImageNet
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('ImageNet: Krzywe Treningu', fontsize=14, fontweight='bold', y=1.00)

for idx, (ax, item) in enumerate(zip(axes.flat, data['imagenet'])):
    history = item['history']
    epochs = [h['epoch'] for h in history]
    train_acc = [h['train_acc'] for h in history]
    test_acc = [h['test_acc'] for h in history]
    loss = [h['loss'] for h in history]
    
    ax2 = ax.twinx()
    
    # Dokładność
    ax.plot(epochs, train_acc, 'o-', label='Train Acc', color='#2E86AB', linewidth=2, markersize=6)
    ax.plot(epochs, test_acc, 's-', label='Test Acc', color='#A23B72', linewidth=2, markersize=6)
    ax.set_ylabel('Accuracy (%)', fontsize=10, fontweight='bold', color='#2E86AB')
    ax.tick_params(axis='y', labelcolor='#2E86AB')
    
    # Loss
    ax2.plot(epochs, loss, '^-', label='Loss', color='#F18F01', linewidth=2, markersize=6)
    ax2.set_ylabel('Loss', fontsize=10, fontweight='bold', color='#F18F01')
    ax2.tick_params(axis='y', labelcolor='#F18F01')
    
    ax.set_xlabel('Epoch', fontsize=10, fontweight='bold')
    ax.set_title(f'{item["model"]} - {item["experiment"]}', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)
    
    # Legenda
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='right', fontsize=8)

plt.tight_layout()
plt.savefig('05_imagenet_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Wnioski Końcowe

### Główne Obserwacje:

1. **CIFAR-10**:
   - ViT_Best osiąga najwyższą dokładność (72.73%)
   - CNN z 10 epokami osiąga 72.04%, bardzo blisko ViT
   - CNN jest znacznie szybszy w treningu
   - ViT_Best wymaga 1240s vs CNN 106s dla 10 epok

2. **ImageNet**:
   - ViT_Best osiąga 66.52%, CNN_10 osiąga 62.37%
   - ViT wyraźnie lepszy na ImageNet
   - Czasy treningu bardziej porównywalne

3. **Liczba Parametrów**:
   - CNN: ~4.3M parametrów (niezmienna)
   - ViT_Small: ~0.4M (najmniej parametrów, najsłabszy)
   - ViT_Best: ~4.7M (porównywalnie z CNN)
   - ViT_Large: ~7.1M (najwięcej parametrów)

4. **Efektywność**:
   - CNN lepszy trade-off na CIFAR-10 (szybko, wysoką dokładność)
   - ViT lepszy na bardziej złożonych danych (ImageNet)
   - Transformery skalują się lepiej z rozmiarom datasetu

### Rekomendacje:
- **CIFAR-10**: CNN jest preferowanym wyborem (szybko, wysoką dokładność)
- **ImageNet**: ViT lepszy, szczególnie ViT_Best
- **Generalnie**: Vision Transformers wykazują większy potencjał na złożonych taskach